In [ ]:
import pandas as pd

In [ ]:
p1_gen=pd.read_csv('Plant_1_Generation_Data.csv')
p2_gen=pd.read_csv('Plant_2_Generation_Data.csv')
p1_weather=pd.read_csv('Plant_1_Weather_Sensor_Data.csv')
p2_weather=pd.read_csv('Plant_2_Weather_Sensor_Data.csv')

In [ ]:
p1_gen.info()
p1_weather.info()
# no null values wow but date_time is object (text)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68778 entries, 0 to 68777
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATE_TIME    68778 non-null  object 
 1   PLANT_ID     68778 non-null  int64  
 2   SOURCE_KEY   68778 non-null  object 
 3   DC_POWER     68778 non-null  float64
 4   AC_POWER     68778 non-null  float64
 5   DAILY_YIELD  68778 non-null  float64
 6   TOTAL_YIELD  68778 non-null  float64
dtypes: float64(4), int64(1), object(2)
memory usage: 3.7+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3182 entries, 0 to 3181
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   DATE_TIME            3182 non-null   object 
 1   PLANT_ID             3182 non-null   int64  
 2   SOURCE_KEY           3182 non-null   object 
 3   AMBIENT_TEMPERATURE  3182 non-null   float64
 4   MODULE_TEMPERATURE   3182 non-null   floa

In [ ]:
p1_gen.head()

,DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD
0,15-05-2020 00:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6259559.0
1,15-05-2020 00:00,4135001,1IF53ai7Xc0U56Y,0.0,0.0,0.0,6183645.0
2,15-05-2020 00:00,4135001,3PZuoBAID5Wc2HD,0.0,0.0,0.0,6987759.0
3,15-05-2020 00:00,4135001,7JYdWkrLSPkdwr4,0.0,0.0,0.0,7602960.0
4,15-05-2020 00:00,4135001,McdE0feGgRqW7Ca,0.0,0.0,0.0,7158964.0


In [ ]:
p1_gen['DATE_TIME'] = pd.to_datetime(p1_gen['DATE_TIME'], errors='coerce')
p1_weather['DATE_TIME'] = pd.to_datetime(p1_weather['DATE_TIME'], errors='coerce')
# coerce means do not give error if not able to convert(wrong format) leave value as NaT
p1_gen.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68778 entries, 0 to 68777
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   DATE_TIME    68778 non-null  datetime64[ns]
 1   PLANT_ID     68778 non-null  int64         
 2   SOURCE_KEY   68778 non-null  object        
 3   DC_POWER     68778 non-null  float64       
 4   AC_POWER     68778 non-null  float64       
 5   DAILY_YIELD  68778 non-null  float64       
 6   TOTAL_YIELD  68778 non-null  float64       
dtypes: datetime64[ns](1), float64(4), int64(1), object(1)
memory usage: 3.7+ MB


/tmp/ipython-input-474/4002243604.py:1: UserWarning: Parsing dates in %d-%m-%Y %H:%M format when dayfirst=False (the default) was specified. Pass `dayfirst=True` or specify a format to silence this warning.
  p1_gen['DATE_TIME'] = pd.to_datetime(p1_gen['DATE_TIME'], errors='coerce')


In [ ]:
#MERGING / JOINING TWO TABLES ON Date_time (common column)
p1_gen.shape
p1_merged = pd.merge(p1_gen,p1_weather,on='DATE_TIME',how='inner') # inner join
p1_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 68774 entries, 0 to 68773
Data columns (total 12 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   DATE_TIME            68774 non-null  datetime64[ns]
 1   PLANT_ID_x           68774 non-null  int64         
 2   SOURCE_KEY_x         68774 non-null  object        
 3   DC_POWER             68774 non-null  float64       
 4   AC_POWER             68774 non-null  float64       
 5   DAILY_YIELD          68774 non-null  float64       
 6   TOTAL_YIELD          68774 non-null  float64       
 7   PLANT_ID_y           68774 non-null  int64         
 8   SOURCE_KEY_y         68774 non-null  object        
 9   AMBIENT_TEMPERATURE  68774 non-null  float64       
 10  MODULE_TEMPERATURE   68774 non-null  float64       
 11  IRRADIATION          68774 non-null  float64       
dtypes: datetime64[ns](1), float64(7), int64(2), object(2)
memory usage: 6.3+ MB


In [ ]:
p1_merged = p1_merged.sort_values('DATE_TIME') # time order matters (past pe future depends)
p1_merged = p1_merged.reset_index(drop=True) # sorted vale index ko original bnadia

In [ ]:
p1_merged['hour'] = p1_merged['DATE_TIME'].dt.hour
p1_merged['day'] = p1_merged['DATE_TIME'].dt.day
p1_merged['month'] = p1_merged['DATE_TIME'].dt.month
# prediction will depend on what season it is (month), night or day (hour) -> peak sunlight and all , day -> early May or late May (weather , sunlight depends )
print(p1_merged[['DATE_TIME','hour','day','month']].head())

   DATE_TIME  hour  day  month
0 2020-05-15     0   15      5
1 2020-05-15     0   15      5
2 2020-05-15     0   15      5
3 2020-05-15     0   15      5
4 2020-05-15     0   15      5


In [ ]:
p1_merged['ac_power_prev_1'] = p1_merged['AC_POWER'].shift(1)
# pichle hour ka power generation btadeta hai same row me .... trend of whether power generation is increasing or decreasing matters
# usually pichli hour ke power generation high hai toh high hoga next hour bhi
# 1 hour lag

In [ ]:
p1_merged['ac_power_prev_24'] = p1_merged['AC_POWER'].shift(24)
# 24 hour lag

In [ ]:
p1_merged = p1_merged.dropna().reset_index(drop=True)
# first row has no previous hour
# first 24 rows have no yesterday value
p1_merged = p1_merged.dropna().reset_index(drop=True)

In [ ]:
y = p1_merged['AC_POWER'] # what we are supposed to predict
X = p1_merged.drop(columns=['DATE_TIME','AC_POWER']) # what we use to predict (date time also not needed now as we have already extracted info from it)

In [ ]:
X = X.drop(columns=['PLANT_ID_x','PLANT_ID_y', 'SOURCE_KEY_x','SOURCE_KEY_y','DC_POWER','DAILY_YIELD','TOTAL_YIELD'],errors='ignore') # we dont require this to train the model
# we want to predict based on only weather + past behavior ....
# dc power and yield directly related to ac power (what we are predicting) THEIR REMOVAL PREVENTS DATA LEAKAGE

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    shuffle=False
)
# shuffle since from older data we predict newer data

In [ ]:
# from sklearn.ensemble import RandomForestRegressor

# rf_model = RandomForestRegressor(
#     n_estimators=200,
#     random_state=42
# )

# rf_model.fit(X_train, y_train)
# Linear Regression not chosen as solar power generation does not behave linearly
# random forest better at predicting complex non linear patterns

In [ ]:
# y_pred = rf_model.predict(X_test)

In [ ]:
# from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
# import numpy as np

# mae = mean_absolute_error(y_test, y_pred) # on avg how wrong is the prediciton
# rmse = np.sqrt(mean_squared_error(y_test, y_pred)) # how big are the errors
# r2 = r2_score(y_test, y_pred) # how well the model explains reality

# print("MAE:", mae)
# print("RMSE:", rmse)
# print("R2:", r2)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np

def run_forecast_pipeline(gen_df, weather_df):

    # Convert datetime
    gen_df['DATE_TIME'] = pd.to_datetime(gen_df['DATE_TIME'], errors='coerce')
    weather_df['DATE_TIME'] = pd.to_datetime(weather_df['DATE_TIME'], errors='coerce')

    # Merge
    df = pd.merge(gen_df, weather_df, on='DATE_TIME', how='inner')

    # Sort
    df = df.sort_values('DATE_TIME').reset_index(drop=True)

    # Time features
    df['hour'] = df['DATE_TIME'].dt.hour
    df['day'] = df['DATE_TIME'].dt.day
    df['month'] = df['DATE_TIME'].dt.month

    # Lag features
    df['ac_power_prev_1'] = df['AC_POWER'].shift(1)
    df['ac_power_prev_24'] = df['AC_POWER'].shift(24)
    df['ac_power_prev_2'] = df['AC_POWER'].shift(2)
    # df['ac_power_prev_3'] = df['AC_POWER'].shift(3)
    # df['ac_power_prev_48'] = df['AC_POWER'].shift(48)

    df['ac_power_roll_3'] = df['AC_POWER'].rolling(3).mean()
    # df['ac_power_roll_24'] = df['AC_POWER'].rolling(24).mean()

    # Drop NaNs from lags
    df = df.dropna().reset_index(drop=True)

    # Target
    y = df['AC_POWER']

    # Features
    X = df.drop(columns = [
        'DATE_TIME','AC_POWER',
        'DC_POWER','DAILY_YIELD','TOTAL_YIELD',
        'PLANT_ID','SOURCE_KEY','SOURCE_KEY_x','SOURCE_KEY_y','PLANT_ID_x','PLANT_ID_y'], errors='ignore')

    # Train-test split
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, shuffle=False
    )

    # Model
    model = RandomForestRegressor(n_estimators=200, max_depth=10,random_state=42)
    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_test)

    # Metrics
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)

    return model,mae, rmse, r2

In [ ]:
p1_results = run_forecast_pipeline(p1_gen, p1_weather)
p2_results = run_forecast_pipeline(p2_gen, p2_weather)

print("Plant 1:", p1_results)
print("Plant 2:", p2_results)
#The forecasting model was plant-sensitive and did not fully generalize across locations.
# so added more lag features , rolling mean and tuned random forest

Plant 1: (8.440196041108372, np.float64(26.341842799727328), 0.994819099004392)
Plant 2: (10.240416179314803, np.float64(24.514383098095372), 0.9925096270691259)


In [ ]:
# test for overfitting